# Time Series Forecasting — Guided Examples

This notebook introduces time-series inspection, naive baselines, leakage-safe lag features, chronological evaluation, and walk-forward validation.

**Important:** The final section evaluates rolling one-step-ahead predictions, where each new observation is assumed to become available before the next forecast. That is different from issuing one fixed multi-day forecast at the start of the test period.

## 1. Imports and reproducible synthetic data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

dates = pd.date_range("2024-01-01", periods=240, freq="D")
t = np.arange(len(dates))
weekly = 9 * np.sin(2 * np.pi * t / 7)
annual_like_slow_wave = 3 * np.sin(2 * np.pi * t / 60)
noise = rng.normal(0, 3, len(dates))
demand = 100 + 0.06 * t + weekly + annual_like_slow_wave + noise

df = pd.DataFrame({"date": dates, "demand": demand})
df.head()

## 2. Inspect timestamps and plot the series

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Rows:", len(df))
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Missing values:\n", df.isna().sum())

ax = df.plot(x="date", y="demand", figsize=(12, 4), title="Daily Demand")
ax.set_ylabel("Demand")
plt.show()

## 3. Chronological train/test split

In [ ]:
HORIZON = 28
train = df.iloc[:-HORIZON].copy()
test = df.iloc[-HORIZON:].copy()

print("Train:", train["date"].min().date(), "to", train["date"].max().date())
print("Test: ", test["date"].min().date(), "to", test["date"].max().date())
assert train["date"].max() < test["date"].min()

## 4. Baseline A — last-value naive forecast

In [ ]:
naive_pred = np.repeat(train["demand"].iloc[-1], len(test))

def forecast_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
    }

naive_metrics = forecast_metrics(test["demand"], naive_pred)
naive_metrics

## 5. Baseline B — seasonal naive forecast

In [ ]:
SEASONAL_PERIOD = 7
last_season = train["demand"].iloc[-SEASONAL_PERIOD:].to_numpy()
seasonal_pred = np.tile(
    last_season, int(np.ceil(len(test) / SEASONAL_PERIOD))
)[:len(test)]

seasonal_metrics = forecast_metrics(test["demand"], seasonal_pred)
seasonal_metrics

## 6. Compare baseline forecasts

In [ ]:
baseline_results = pd.DataFrame([
    {"model": "naive", **naive_metrics},
    {"model": "seasonal_naive_7", **seasonal_metrics},
])
baseline_results

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train["date"].tail(45), train["demand"].tail(45), label="Recent training data")
plt.plot(test["date"], test["demand"], label="Actual test demand")
plt.plot(test["date"], naive_pred, label="Naive forecast")
plt.plot(test["date"], seasonal_pred, label="Seasonal-naive forecast")
plt.title("Baseline Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Create leakage-safe lag and calendar features

In [ ]:
features = df.copy()
features["lag_1"] = features["demand"].shift(1)
features["lag_7"] = features["demand"].shift(7)
# Shift first so the current row's target is not included in its rolling mean.
features["rolling_mean_7"] = features["demand"].shift(1).rolling(7).mean()
features["day_of_week"] = features["date"].dt.dayofweek
features = features.dropna().reset_index(drop=True)

feature_cols = ["lag_1", "lag_7", "rolling_mean_7", "day_of_week"]
features[["date", "demand"] + feature_cols].head()

## 8. Fit a regression model with a chronological holdout

In [ ]:
# This is a rolling one-step-ahead evaluation: lag values for each test day
# use observations that would be known by that day.
train_f = features.iloc[:-HORIZON].copy()
test_f = features.iloc[-HORIZON:].copy()

model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(train_f[feature_cols], train_f["demand"])
rf_pred = model.predict(test_f[feature_cols])

rf_metrics = forecast_metrics(test_f["demand"], rf_pred)
rf_metrics

## 9. Compare all models on the same test dates

In [ ]:
results = pd.DataFrame([
    {"model": "naive", **naive_metrics},
    {"model": "seasonal_naive_7", **seasonal_metrics},
    {"model": "random_forest_lags", **rf_metrics},
]).sort_values("MAE", ignore_index=True)

results

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_f["date"], test_f["demand"], marker="o", label="Actual")
plt.plot(test_f["date"], naive_pred, label="Naive")
plt.plot(test_f["date"], seasonal_pred, label="Seasonal naive")
plt.plot(test_f["date"], rf_pred, label="Random forest")
plt.title("Forecasts on the Holdout Period")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Walk-forward validation with TimeSeriesSplit

In [ ]:
X = features[feature_cols]
y = features["demand"]
tscv = TimeSeriesSplit(n_splits=5)
fold_rows = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), start=1):
    fold_model = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    fold_model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    val_pred = fold_model.predict(X.iloc[val_idx])
    fold_rows.append({
        "fold": fold,
        "train_end": features["date"].iloc[tr_idx[-1]].date(),
        "validation_start": features["date"].iloc[val_idx[0]].date(),
        "validation_end": features["date"].iloc[val_idx[-1]].date(),
        "MAE": mean_absolute_error(y.iloc[val_idx], val_pred),
        "RMSE": np.sqrt(mean_squared_error(y.iloc[val_idx], val_pred)),
    })

fold_results = pd.DataFrame(fold_rows)
display(fold_results)
print("Mean fold MAE:", round(fold_results["MAE"].mean(), 3))
print("Std. dev. fold MAE:", round(fold_results["MAE"].std(), 3))

## 11. MAPE caution

MAPE divides each absolute error by the actual value. It is undefined when an actual value is zero and unstable near zero. Prefer MAE/RMSE or a carefully chosen scaled metric when zeros are possible.

## 12. Final checklist

- Did the split preserve time order?
- Were lag and rolling features built from past values only?
- Were all models scored on the same test timestamps?
- Does the evaluation simulate the real forecast horizon?
- Is the model better than a simple baseline?
- Are uncertainty and metric limitations documented?

### Suggested extension

Add a known-in-advance calendar feature such as `is_weekend`, then compare it with the current model. For a fixed multi-step forecast, redesign the prediction process so actual test-period targets are never used as lag inputs.